# Daisy Peri T - Peristaltic Pump Quickstart Guide

By the end of this notebook you will know how to find your calibration flow rate (at 60 rpm), set your calibration value, and run your **Daisy Peri T** peristaltic pump. 

## Find the calibration value for your Daisy Peristaltic pump
- First, we measure the volume delivered to find the calibration flow rate at 60 rpm. Calibration tells the Daisy library how much liquid your tubing actually delivers at 60 rpm. Daisy Peristaltic pumps are generally linear across their rpm range, meaning a single calibration point at 60 rpm is a reasonable approximation for **most** applications.
- However, real-world flow rate can deviate from linearity especially due to tubing elasticity, back-pressure, and roller dynamics.
- To find if your application requires a non-linear model, please refer to the **Daisy Peri T  - Calibration Curve Fitting** guide, which measures flow rate at multiple rpm setpoints and finds a best-fit curve (linear, quadratic, or power law) to predict flow rate more accurately at any target rpm.


## Calibrate your pump

- Set your measured flow rate at 60 rpm by using `pump.calibrate()`

## Learn how to run your pump

- Use `pump.run(flow_rate, volume, wait=False)` to run your pump with parameters:
>    `flow_rate` — flow rate in ml/min \
>    `volume` — volume to dispense in ml, a negative value directs the pump to run in reverse \
>    `wait` — if `True`, waits for dispensing to complete before running next line
- Estimate the run time without running the pump by using `pump.estimate_run_time(flow_rate, volume)`


## Materials:
- Your Daisy Peri T pump and liquid in the configuration in which you will run your application
- A scale, preferably measuring to 0.1 g
- A vessel to collect the dispensed liquid

### Peri T's reference chart for calibration flow rates based on tubing size and number of rollers.

You can use this reference chart to find the expected calibration flow rate at 60 rpm based on your tubing size and the number of rollers in your Peri T.

| Tubing ID / OD (mm) | Flow rate (ml/min) for a 3 Roller|Flow rate (ml/min) for a 6 Roller|
|---------------------|-----------------------------|-----------------------------|
| 0.8 / 4.0           | 2.01                        |1.87                         |
| 1.6 / 4.8           | 11.22                       |7.54                         |
| 2.4 / 5.6           | 22.23                       |12.90                        |
| 3.2 / 6.4           | 38.15                       |20.70                        |
| 4.8 / 8.0           | 73.14                       |38.16                        |

## What affects your calibration number

The tables above are a reference — your measured value may differ due to:

- **Fluid viscosity** — thicker liquids deliver less volume per rotation than water
- **Tubing wear** — tubing stretches over time and flow rate will drift, so recalibrate if tubing has been in use for a while
- **Tubing installation** — how tightly the tubing is seated in the pump head affects compression and flow
- **Temperature** — affects both fluid viscosity and tubing elasticity
- **Application setup** - outlet height, back-pressure, submersion depth, and any restriction at the tubing outlet (such as a needle, nozzle, or narrow tip) all affect the actual volume delivered.

**Always calibrate with the same liquid and application set up you will use in your protocol.**

## Import
The Daisy Python Library is already loaded into the Daisy Controller Software, all you have to do is import any external libraries you need. Here we are importing time to allow for pauses before running the pump by using `time.sleep()`.

In [ ]:
import time

## Print instruments found
Print the instruments found to easily verify the index of the Daisy you will be using.

In [ ]:
print("Instruments found:")
for position, (code, inst) in enumerate(zip(self.my_interface.instruments_list[0], self.my_interface.groups[0].inst)):
    print(f"  [{position}]  {code} — {inst.name}")

## Set your instrument variable name
Change `INST_INDEX` to the position of the Daisy (Peri T) you want to use.

In [ ]:
# Set which instrument you are calibrating
INST_INDEX = 0   # 0 if first instrument or change to match your pump's position

pump = self.my_interface.groups[0].inst[INST_INDEX]
print(f"Calibrating: {pump.name}")
print(f"Current calibration: {pump.calibration_speed} ml/min @ 60 RPM")

## Prime the tubing

Before measuring, make sure the tubing is fully primed with no air bubbles.

Set `prime_volume` based on your **tubing size** and **length**. The cell below contains a starting point of 10 ml for priming a Peri T. Change the volume depending on the amount needed to prime your tubing.

**Before running the next cell,** familiarize yourself with how to stop the pump to not unnecessarily waste liquid (found in the next markdown cell).

In [ ]:
# prime_volume starting point for Peri T
# Adjust based on your tubing size and length
prime_volume = 10.0    # In ml

pump.run(pump.calibration_speed, prime_volume, wait=True)
print("Confirm liquid has primed all of the tubing.")

## **To stop the pump run:** 

Use the Jupyter stop button (■) in the toolbar (or press `I` twice to interrupt the kernel) to stop the `pump.run()` function above. Then run the next cell.

In [ ]:
pump.stop()

## Run the calibration measurement

Check the density of your liquid. If your liquid has the same density as water (1g/ml), directly enter the measured grams as your volume.

Tare collection vessel on the scale then place under the pump outlet. The next cell runs the pump at 60 rpm for exactly 60 seconds, then stops automatically. Weigh the collection vessel to find the volume of liquid dispensed.

**Do not interrupt the next cell** — the full 60 seconds is needed for an accurate measurement.

In [ ]:
print("Place your collection vessel under the tubing outlet.")
print("Starting in 5 seconds...")
time.sleep(5)

print("\nRunning...")
pump.run(pump.calibration_speed, 999.0, wait=False)

for remaining in range(60, 0, -1):
    print(f"  {remaining}s remaining...", end='\r')
    time.sleep(1)

pump.stop()
print("\nPump stopped. Measure the liquid in your collection vessel.")

## Enter your measured volume and apply calibration

Enter the volume (ml) you collected and run this cell to apply the calibration.

In [ ]:
MEASURED_VOLUME_ML = 0.0   # replace with your measured value in ml

pump.calibrate(MEASURED_VOLUME_ML)

print(f"Calibration applied: {pump.calibration_speed} ml/min @ 60 RPM")
print(f"\nTo use this calibration in another notebook:")
print(f"  pump.calibrate({pump.calibration_speed})")

## Calculate the **max** flow rate with your calculated calibration
Peri T's max speed is 300 rpm

In [ ]:
# Peri Ts have a max rpm of 300
max_rpm = 300

max_flow_rate = (max_rpm / 60) * pump.calibration_speed

print(f"With Peri T calibrated at {pump.calibration_speed} ml/min, the max flow rate you can run is: {max_flow_rate} ml/min")

## Run your pump

Use `pump.run(flow_rate, volume, wait=False)` to run your pump with parameters:
>    `flow_rate` — flow rate in ml/min \
>    `volume` — volume to dispense in ml, a negative value directs the pump to run in reverse \
>    `wait` — if `True`, waits for dispensing to complete before running next line

## Example run arguments for Peri T

**To stop the pump:** Use the Jupyter stop button (■) in the toolbar (or press `I` twice to interrupt the kernel) to stop the `pump.run()` function. Then run `pump.run()`.

In [ ]:
# Sets pump flow rate to 30 ml/min, dispenses 2 ml, pump finishes running before printing
pump.run(30.0, 2.0, True)
print("Pump has finished running.")

In [ ]:
pump.stop()

In [ ]:
# Estimate the run time of the Peri T with these parameters
print(pump.estimate_run_time(30.0, 2.0))

In [ ]:
# Sets pump flow rate to 30 ml/min, dispenses 2 ml, pump does not finish running before printing
pump.run(30.0, 2.0, False)
print("Pump has not finished running yet.")

In [ ]:
pump.stop()

In [ ]:
# Sets pump flow rate to 30 ml/min, aspirates 2 ml
# Negative value indicates the pump to run counter clockwise
pump.run(30.0, -2.0, True)